In [15]:
import torch
import torchvision
from sklearn.metrics import f1_score as sklearn_f1_score
from sklearn.metrics import precision_score, recall_score, accuracy_score
from sklearn.preprocessing import StandardScaler

import torch.nn as nn
import torch.optim as optim
import copy


import tqdm
import sys, os, random
sys.path.append(os.path.join(os.getcwd(), 'CONFOLD')) #add CONFOLD to path

import numpy as np
from sklearn.metrics import accuracy_score


import pandas as pd
import matplotlib.pyplot as plt

from foldrm import Classifier
from datasets import  final_extinctionrisk_dataframe, final_extinctionrisk_noth_dataframe

In [16]:
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "No GPU found")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

GPU available: True
GPU name: NVIDIA GeForce RTX 4090
Using device: cuda


In [17]:
random.seed(42)

# HUMAN THREATS Included

In [18]:
model, data = final_extinctionrisk_dataframe()

categorical_data = data.drop(model.numeric, axis=1)
categorical_data_without_y = categorical_data.drop(model.label, axis=1)
categorical_data_without_y_dummies = pd.get_dummies(categorical_data_without_y, dtype=int)
one_hot_dataset = pd.concat([data[model.numeric], categorical_data_without_y_dummies],axis=1)

X = one_hot_dataset

X = X.to_numpy()
mapping = {'Lower_risk':0, 'Higher_risk':1}
y = data[model.label]
y = y.map(mapping)
y = y.to_numpy()

X = torch.tensor(X,dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

X.to(device)
y.to(device)

tensor([0., 1., 1.,  ..., 0., 0., 0.], device='cuda:0')

In [19]:
import csv

train_indices = []
test_indices = []
for i in range(1,11):
    with open("./datasets/r_folds/fold_"+str(i)+"_train_idx.csv", newline='') as csvfile:
        train_df = pd.read_csv(csvfile)
        train_list = train_df.to_numpy().flatten().tolist()
        train_indices.append(train_list)
    with open("./datasets/r_folds/fold_"+str(i)+"_test_idx.csv", newline='') as csvfile:
        test_df = pd.read_csv(csvfile)
        test_list = train_df.to_numpy().flatten().tolist()
        test_indices.append(test_list)

In [20]:
def model_train(model, X_train, y_train, X_val, y_val):
    #writer = SummaryWriter(filename_suffix=model.name)
    # loss function and optimizer
    loss_fn = nn.BCELoss()  # binary cross entropy
    optimizer = optim.Adam(model.parameters(), lr=0.0001)
 
    n_epochs = 50   # number of epochs to run
    batch_size = 256  # size of each batch
    batch_start = torch.arange(0, len(X_train), batch_size)
 
    # Hold the best model
    best_acc = - np.inf   # init to negative infinity
    best_rec = - np.inf   # init to negative infinity
    best_pre = - np.inf   # init to negative infinity
    best_f1 = - np.inf   # init to negative infinity
    best_weights = None
 
    for epoch in range(n_epochs):
        model.train()
        with tqdm.tqdm(batch_start, unit="batch", mininterval=0, disable=True) as bar:
            bar.set_description(f"Epoch {epoch}")
            acc_sum = 0
            loss_sum = 0
            for start in bar:
                # take a batch
                X_batch = X_train[start:start+batch_size]
                y_batch = y_train[start:start+batch_size].unsqueeze(dim=1)

                mean = X_batch.mean(dim=0)
                std = X_batch.std(dim=0, unbiased=False)
                # forward pass
                y_pred = model(X_batch)
                loss = loss_fn(y_pred, y_batch)
                # backward pass
                optimizer.zero_grad()
                loss.backward()
                # update weights
                optimizer.step()
                # print progress
                acc = (y_pred.round() == y_batch).float().mean()
                
                bar.set_postfix(
                    loss=float(loss),
                    acc=float(acc)
                )
                acc_sum = acc_sum+acc
                loss_sum = loss_sum+loss
            #writer.add_scalar("train/loss", loss_sum/bar.total, epoch)
            #writer.add_scalar("train/acc",acc_sum/bar.total, epoch)
        # evaluate accuracy at end of each epoch
        model.eval()
        y_pred = model(X_val)
        preds_cpu = y_pred.cpu()
        labels_cpu = y_val.cpu()
        preds_numpy = np.round(preds_cpu.detach().numpy().flatten())
        labels_numpy = labels_cpu.detach().numpy()
        loss = loss_fn(labels_cpu, torch.flatten(preds_cpu))
        #writer.add_scalar("test/loss", loss, epoch)
        f1 = sklearn_f1_score(labels_numpy, preds_numpy, average=None, labels=[1, 0])
        acc = accuracy_score(labels_numpy, preds_numpy)
        #writer.add_scalar("test/acc", acc, epoch)
        precision = precision_score(labels_numpy, preds_numpy, average=None, labels=[1, 0])
        recall = recall_score(labels_numpy, preds_numpy, average=None, labels=[1, 0])
        
        if acc > best_acc:
            best_acc = acc
            best_weights = copy.deepcopy(model.state_dict())
            best_rec = recall
            best_pre = precision
            best_f1 = f1
            stop_state = epoch
    # restore model and return best accuracy
    model.load_state_dict(best_weights)
    #writer.flush()
    return best_acc, best_rec, best_pre, best_f1, stop_state

In [ ]:
class SLP(torch.nn.Module):
    def __init__(self, input_size):
        super(SLP, self).__init__()
        self.linear1 = torch.nn.Linear(input_size, 1)
        self.sigmoid = torch.nn.Sigmoid()
        self.name = "SLP"

    def forward(self, x):
        x = self.sigmoid(self.linear1(x))
        return x
    
# create model, train, and get accuracy



acc_metrics = []
f1_metrics = [] 
precision_metrics = []
recall_metrics = []
for fold, (train_idx, val_idx) in enumerate(zip(train_indices, test_indices)):
    print("Fold"+str(fold))
    scaler = StandardScaler()
    model = SLP(np.shape(X)[1]).to(device)
    x_train = scaler.fit_transform(X[train_idx])
    y_train = y[train_idx].to(device)
    x_val = scaler.fit_transform(X[val_idx])
    x_train = torch.tensor(x_train, dtype=torch.float32).to(device)
    x_val = torch.tensor(x_val, dtype=torch.float32).to(device)
    y_val = y[val_idx].to(device)
    acc, recall, precision, f1, epoch_num = model_train(model, x_train, y_train, x_val, y_val)
    acc_metrics.append(acc)
    f1_metrics.append(f1)
    precision_metrics.append(precision)
    recall_metrics.append(recall)
print("Single Layer Perceptron, Including Human Threats")
print(f"KFold Mean Acc: {np.mean(acc_metrics)} | KFold SD: {np.std(acc_metrics)}")
f1_metrics = np.array(f1_metrics)
precision_metrics = np.array(precision_metrics)
recall_metrics = np.array(recall_metrics)
print(f"KFold Mean f1: {np.mean(f1_metrics[:,0]),np.mean(f1_metrics[:,1])} | KFold SD: {np.std(f1_metrics[:,0]), np.std(f1_metrics[:,1])}")
print(f"KFold Mean Precision: {np.mean(precision_metrics[:, 0]), np.mean(precision_metrics[:, 1])} | KFold SD: {np.std(precision_metrics[:, 0]), np.std(precision_metrics[:, 1])}")
print(f"KFold Mean Recall: {np.mean(recall_metrics[:, 0]), np.mean(recall_metrics[:, 1])} | KFold SD: {np.std(recall_metrics[:, 0]), np.std(recall_metrics[:, 1])}")

Fold0


NameError: name 'X_val' is not defined

In [ ]:
class TwoLayerMLPwithRELU(torch.nn.Module):
    def __init__(self, input_size):
        super(TwoLayerMLPwithRELU, self).__init__()
        self.linear1 = torch.nn.Linear(input_size, 256)
        self.activation1 = torch.nn.ReLU()
        #self.dropout1 = nn.Dropout(0.2)
        self.linear2 = torch.nn.Linear(256, 1)
        self.sigmoid = torch.nn.Sigmoid()
        self.name = "TwoLayerMLPwithRELU"

    def forward(self, x):
        x = self.activation1(self.linear1(x))
        #x = self.dropout1(x)
        x = self.sigmoid(self.linear2(x))
        return x

# create model, train, and get accuracy
model = TwoLayerMLPwithRELU(np.shape(X)[1])
model.to(device)

acc_metrics = []
f1_metrics = [] 
precision_metrics = []
recall_metrics = []
for fold, (train_idx, val_idx) in enumerate(zip(train_indices, test_indices)):
    print("Fold"+str(fold))
    scaler = StandardScaler()
    model = SLP(np.shape(X)[1]).to(device)
    x_train = scaler.fit_transform(X[train_idx])
    y_train = y[train_idx].to(device)
    x_val = scaler.fit_transform(X[val_idx])
    x_train = torch.tensor(x_train, dtype=torch.float32).to(device)
    x_val = torch.tensor(x_val, dtype=torch.float32).to(device)
    y_val = y[val_idx].to(device)
    acc, recall, precision, f1, epoch_num = model_train(model, x_train, y_train, x_val, y_val)
    acc_metrics.append(acc)
    f1_metrics.append(f1)
    precision_metrics.append(precision)
    recall_metrics.append(recall)
print("Two Layer Perceptron, Including Human Threats")
print(f"KFold Mean Acc: {np.mean(acc_metrics)} | KFold SD: {np.std(acc_metrics)}")
f1_metrics = np.array(f1_metrics)
precision_metrics = np.array(precision_metrics)
recall_metrics = np.array(recall_metrics)
print(f"KFold Mean f1: {np.mean(f1_metrics[:,0]),np.mean(f1_metrics[:,1])} | KFold SD: {np.std(f1_metrics[:,0]), np.std(f1_metrics[:,1])}")
print(f"KFold Mean Precision: {np.mean(precision_metrics[:, 0]), np.mean(precision_metrics[:, 1])} | KFold SD: {np.std(precision_metrics[:, 0]), np.std(precision_metrics[:, 1])}")
print(f"KFold Mean Recall: {np.mean(recall_metrics[:, 0]), np.mean(recall_metrics[:, 1])} | KFold SD: {np.std(recall_metrics[:, 0]), np.std(recall_metrics[:, 1])}")

In [ ]:
class FiveLayerMLPwithRELU(torch.nn.Module):
    def __init__(self, input_size):
        super(FiveLayerMLPwithRELU, self).__init__()
        self.linear1 = torch.nn.Linear(input_size, 256)
        self.activation1 = torch.nn.ReLU()
        self.dropout1 = nn.Dropout(0.2)
        self.linear2 = torch.nn.Linear(256, 256)
        self.activation2 = torch.nn.ReLU()
        #self.dropout2 = nn.Dropout(0.2)
        self.linear3 = torch.nn.Linear(256, 256)
        self.activation3 = torch.nn.ReLU()
        #self.dropout3 = nn.Dropout(0.2)
        self.linear4 = torch.nn.Linear(256, 256)
        self.activation4 = torch.nn.ReLU()
        #self.dropout4 = nn.Dropout(0.2)
        self.linear5 = torch.nn.Linear(256, 1)
        self.sigmoid = torch.nn.Sigmoid()
        self.name = "FiveLayerMLPwithRELU"

    def forward(self, x):
        x = self.activation1(self.linear1(x))
        #x = self.dropout1(x)
        x = self.activation2(self.linear2(x))
        #x = self.dropout2(x)
        x = self.activation3(self.linear3(x))
        #x = self.dropout3(x)
        x = self.activation4(self.linear4(x))
        #x = self.dropout4(x)
        x = self.sigmoid(self.linear5(x))
        return x

# create model, train, and get accuracy
model = FiveLayerMLPwithRELU(np.shape(X)[1])
model.to(device)

acc_metrics = []
f1_metrics = [] 
precision_metrics = []
recall_metrics = []
for fold, (train_idx, val_idx) in enumerate(zip(train_indices, test_indices)):
    print("Fold"+str(fold))
    scaler = StandardScaler()
    model = SLP(np.shape(X)[1]).to(device)
    x_train = scaler.fit_transform(X[train_idx])
    y_train = y[train_idx].to(device)
    x_val = scaler.fit_transform(X[val_idx])
    x_train = torch.tensor(x_train, dtype=torch.float32).to(device)
    x_val = torch.tensor(x_val, dtype=torch.float32).to(device)
    y_val = y[val_idx].to(device)
    acc, recall, precision, f1, epoch_num = model_train(model, x_train, y_train, x_val, y_val)
    acc_metrics.append(acc)
    f1_metrics.append(f1)
    precision_metrics.append(precision)
    recall_metrics.append(recall)
print("Five Layer Perceptron, Including Human Threats")
print(f"KFold Mean Acc: {np.mean(acc_metrics)} | KFold SD: {np.std(acc_metrics)}")
f1_metrics = np.array(f1_metrics)
precision_metrics = np.array(precision_metrics)
recall_metrics = np.array(recall_metrics)
print(f"KFold Mean f1: {np.mean(f1_metrics[:,0]),np.mean(f1_metrics[:,1])} | KFold SD: {np.std(f1_metrics[:,0]), np.std(f1_metrics[:,1])}")
print(f"KFold Mean Precision: {np.mean(precision_metrics[:, 0]), np.mean(precision_metrics[:, 1])} | KFold SD: {np.std(precision_metrics[:, 0]), np.std(precision_metrics[:, 1])}")
print(f"KFold Mean Recall: {np.mean(recall_metrics[:, 0]), np.mean(recall_metrics[:, 1])} | KFold SD: {np.std(recall_metrics[:, 0]), np.std(recall_metrics[:, 1])}")